# Lab Task:

####  Model: Build and train a custom CNN with the following structure:

Conv2D(64 filters, (3×3), ReLU) → BatchNormalization → MaxPooling2D(2×2)

Conv2D(128 filters, (3×3), ReLU) → BatchNormalization → MaxPooling2D(2×2)

Conv2D(256 filters, (3×3), ReLU) → Dropout(0.3) → MaxPooling2D(2×2)

Flatten → Dense(256, ReLU) → Dropout(0.5) → Dense(1, Sigmoid)

#### Dataset: Use only two classes:

from the last lab tasks

#### Compile with: binary_crossentropy loss and adam optimizer.

#### Train and evaluate the model.

In [ ]:
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras import optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import seaborn as sns

In [ ]:
checkpoints = r'C:\Users\Admin\Downloads\Computer vision\checkpoint\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'

In [3]:
train_dir = r'C:\Users\Admin\Downloads\Computer vision\train'
validation_dir = r'C:\Users\Admin\Downloads\Computer vision\validation'
test_dir = r'C:\Users\Admin\Downloads\Computer vision\test'

In [4]:
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu',input_shape=(256, 256, 3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Flatten())
model.add(layers.Dense(512, activation='relu'))
model.add(layers.Dense(4,activation='softmax'))

C:\Users\Admin\anaconda3\envs\dsp\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 254, 254, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 127, 127, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 125, 125, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 62, 62, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 60, 60, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 30, 30, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 28, 28, 128)         │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 14, 14, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 25088)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 512)                 │      12,845,568 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 4)                   │           2,052 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 13,088,452 (49.93 MB)

 Trainable params: 13,088,452 (49.93 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.compile(loss='categorical_crossentropy', optimizer=optimizers.RMSprop(learning_rate=1e-4), metrics=['acc'])

In [9]:
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(train_dir, target_size=(256, 256), batch_size=32,class_mode='categorical')
validation_generator = test_datagen.flow_from_directory(validation_dir,target_size=(256, 256),batch_size=32,class_mode='categorical')

Found 1600 images belonging to 4 classes.
Found 252 images belonging to 4 classes.


In [10]:
EpochCheckpoint = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
callbacks = [EpochCheckpoint]

In [ ]:
step_size_train = train_generator.n//train_generator.batch_size

model_history = model.fit(train_generator,
                    validation_data =validation_generator,
                   steps_per_epoch=step_size_train,
                   epochs=100,
                    callbacks=callbacks)

C:\Users\Admin\anaconda3\envs\dsp\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - acc: 0.3884 - loss: 1.2864
Epoch 1: val_loss improved from inf to 0.97440, saving model to C:\Users\Admin\Downloads\Computer vision\checkpoint\E1-cp-0001-loss0.97.h5


50/50 ━━━━━━━━━━━━━━━━━━━━ 129s 3s/step - acc: 0.3909 - loss: 1.2826 - val_acc: 0.5754 - val_loss: 0.9744
Epoch 2/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - acc: 0.7079 - loss: 0.6782
Epoch 2: val_loss improved from 0.97440 to 0.44847, saving model to C:\Users\Admin\Downloads\Computer vision\checkpoint\E1-cp-0002-loss0.45.h5


50/50 ━━━━━━━━━━━━━━━━━━━━ 106s 2s/step - acc: 0.7084 - loss: 0.6767 - val_acc: 0.8016 - val_loss: 0.4485
Epoch 3/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - acc: 0.7806 - loss: 0.4629
Epoch 3: val_loss improved from 0.44847 to 0.37456, saving model to C:\Users\Admin\Downloads\Computer vision\checkpoint\E1-cp-0003-loss0.37.h5


50/50 ━━━━━━━━━━━━━━━━━━━━ 153s 3s/step - acc: 0.7808 - loss: 0.4627 - val_acc: 0.8056 - val_loss: 0.3746
Epoch 4/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - acc: 0.8248 - loss: 0.4372
Epoch 4: val_loss did not improve from 0.37456
50/50 ━━━━━━━━━━━━━━━━━━━━ 214s 4s/step - acc: 0.8247 - loss: 0.4367 - val_acc: 0.7857 - val_loss: 0.3885
Epoch 5/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - acc: 0.8356 - loss: 0.3546
Epoch 5: val_loss improved from 0.37456 to 0.31817, saving model to C:\Users\Admin\Downloads\Computer vision\checkpoint\E1-cp-0005-loss0.32.h5


50/50 ━━━━━━━━━━━━━━━━━━━━ 239s 5s/step - acc: 0.8357 - loss: 0.3545 - val_acc: 0.8532 - val_loss: 0.3182
Epoch 6/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - acc: 0.8701 - loss: 0.2966
Epoch 6: val_loss did not improve from 0.31817
50/50 ━━━━━━━━━━━━━━━━━━━━ 241s 5s/step - acc: 0.8698 - loss: 0.2968 - val_acc: 0.8095 - val_loss: 0.3716
Epoch 7/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - acc: 0.8711 - loss: 0.2856
Epoch 7: val_loss did not improve from 0.31817
50/50 ━━━━━━━━━━━━━━━━━━━━ 249s 5s/step - acc: 0.8712 - loss: 0.2855 - val_acc: 0.8254 - val_loss: 0.3264
Epoch 8/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - acc: 0.8805 - loss: 0.2627
Epoch 8: val_loss improved from 0.31817 to 0.26785, saving model to C:\Users\Admin\Downloads\Computer vision\checkpoint\E1-cp-0008-loss0.27.h5


50/50 ━━━━━━━━━━━━━━━━━━━━ 221s 4s/step - acc: 0.8806 - loss: 0.2625 - val_acc: 0.8770 - val_loss: 0.2679
Epoch 9/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - acc: 0.8976 - loss: 0.2301
Epoch 9: val_loss improved from 0.26785 to 0.25880, saving model to C:\Users\Admin\Downloads\Computer vision\checkpoint\E1-cp-0009-loss0.26.h5


50/50 ━━━━━━━━━━━━━━━━━━━━ 148s 3s/step - acc: 0.8976 - loss: 0.2301 - val_acc: 0.8810 - val_loss: 0.2588
Epoch 10/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - acc: 0.9294 - loss: 0.1783
Epoch 11: val_loss did not improve from 0.23147
50/50 ━━━━━━━━━━━━━━━━━━━━ 159s 3s/step - acc: 0.9292 - loss: 0.1786 - val_acc: 0.8730 - val_loss: 0.2911
Epoch 12/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - acc: 0.9288 - loss: 0.1701
Epoch 12: val_loss did not improve from 0.23147
50/50 ━━━━━━━━━━━━━━━━━━━━ 142s 3s/step - acc: 0.9288 - loss: 0.1702 - val_acc: 0.8849 - val_loss: 0.2846
Epoch 13/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - acc: 0.9281 - loss: 0.1681
Epoch 13: val_loss did not improve from 0.23147
50/50 ━━━━━━━━━━━━━━━━━━━━ 137s 3s/step - acc: 0.9282 - loss: 0.1679 - val_acc: 0.8929 - val_loss: 0.2427
Epoch 14/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - acc: 0.9479 - loss: 0.1393
Epoch 14: val_loss improved from 0.23147 to 0.21323, saving model to C:\Users\Admin\Downloads\Computer vision\ch

50/50 ━━━━━━━━━━━━━━━━━━━━ 150s 3s/step - acc: 0.9478 - loss: 0.1393 - val_acc: 0.9008 - val_loss: 0.2132
Epoch 15/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - acc: 0.9578 - loss: 0.1158
Epoch 15: val_loss did not improve from 0.21323
50/50 ━━━━━━━━━━━━━━━━━━━━ 136s 3s/step - acc: 0.9576 - loss: 0.1161 - val_acc: 0.8492 - val_loss: 0.4239
Epoch 16/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - acc: 0.9500 - loss: 0.1334
Epoch 16: val_loss did not improve from 0.21323
50/50 ━━━━━━━━━━━━━━━━━━━━ 127s 3s/step - acc: 0.9501 - loss: 0.1332 - val_acc: 0.9048 - val_loss: 0.2726
Epoch 17/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - acc: 0.9608 - loss: 0.0966
Epoch 17: val_loss improved from 0.21323 to 0.20044, saving model to C:\Users\Admin\Downloads\Computer vision\checkpoint\E1-cp-0017-loss0.20.h5


50/50 ━━━━━━━━━━━━━━━━━━━━ 104s 2s/step - acc: 0.9608 - loss: 0.0967 - val_acc: 0.9206 - val_loss: 0.2004
Epoch 18/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - acc: 0.9617 - loss: 0.0992
Epoch 18: val_loss did not improve from 0.20044
50/50 ━━━━━━━━━━━━━━━━━━━━ 101s 2s/step - acc: 0.9617 - loss: 0.0992 - val_acc: 0.9087 - val_loss: 0.2059
Epoch 19/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - acc: 0.9689 - loss: 0.0857
Epoch 19: val_loss did not improve from 0.20044
50/50 ━━━━━━━━━━━━━━━━━━━━ 102s 2s/step - acc: 0.9690 - loss: 0.0856 - val_acc: 0.8770 - val_loss: 0.3800
Epoch 20/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - acc: 0.9651 - loss: 0.0862
Epoch 20: val_loss did not improve from 0.20044
50/50 ━━━━━━━━━━━━━━━━━━━━ 117s 2s/step - acc: 0.9651 - loss: 0.0862 - val_acc: 0.9246 - val_loss: 0.2279
Epoch 21/100
36/50 ━━━━━━━━━━━━━━━━━━━━ 28s 2s/step - acc: 0.9850 - loss: 0.0550

In [ ]:
model_history.history

In [ ]:
acc = model_history.history['acc']
val_acc = model_history.history['val_acc']
loss = model_history.history['loss']
val_loss = model_history.history['val_loss']
epochs = range(1, len(acc) + 1)
plt.plot(epochs, acc, 'bo', label='Training acc')
plt.plot(epochs, val_acc, 'b', label='Validation acc')
plt.title('Training and validation accuracy')
plt.legend()
plt.figure()
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend()
plt.show()
plt.savefig(r'C:\Users\Administrator\Downloads\ML Lab\Computer vision\lab11\model_Accuracy.png')

In [ ]:
model.save(r'C:\Users\Administrator\Downloads\ML Lab\Computer vision\lab12\E1-cp-0012-loss0.21.h5')

In [ ]:
#model = load_model(r'C:\Users\Administrator\Downloads\ML Lab\Computer vision\lab11\model_lab11.h5')
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(test_dir, target_size=(256, 256), batch_size=32, shuffle=False, class_mode='categorical')
label=test_generator.labels
preds=model.predict(test_generator)
pred = np.argmax(preds, axis = 1)
cm = confusion_matrix(label, pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,  display_labels=['Cercospora', 'common_rust','healthy', 'leaf_blight'])
disp.plot()
plt.show()
#plt.savefig(r'C:\Users\Administrator\Downloads\ML Lab\Computer vision\lab11\confusion_matrics.jpg')

In [ ]:
print(classification_report(label, pred, target_names=['Cercospora', 'common_rust','healthy', 'leaf_blight']))